# 🌱 EcoDyePredict: Machine Learning Modeling of Dye Removal by Pumpkin Seed Husk (PSH)

### Project Overview
This notebook provides end-to-end Exploratory Data Analysis (EDA), model benchmarking, and operational optimization for predicting the removal efficiency (%) of synthetic dyes from wastewater using **Pumpkin Seed Husk (PSH)** (`Cucurbita pepo`) as an eco-friendly biosorbent.

**Key Input Parameters:**
1. `pH` (2.0 - 12.0)
2. `dosage_g_L` (Adsorbent dose in g/L)
3. `conc_mg_L` (Initial dye concentration in mg/L)
4. `time_min` (Contact time in minutes)
5. `temp_C` (Operating temperature in °C)

**Target:** `removal_pct` (Dye removal efficiency %)

In [ ]:
import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set_theme(style="whitegrid", font_scale=1.1)
ROOT = pathlib.Path("..")
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
print("Environment initialized successfully.")

## 1. Load Model Registry & Trained Benchmarks

In [ ]:
reg_file = MODELS_DIR / "dye_registry.json"
if reg_file.exists():
    registry = json.loads(reg_file.read_text())
    reg_df = pd.DataFrame([
        {"Dye": k.title(), "Rows": v["rows"], "Best Model": v["best_model"], "Test R²": round(v["test_r2"], 3)}
        for k, v in registry.items()
    ])
    display(reg_df)
else:
    print("Registry file not found. Run train_all_dyes.py first.")

## 2. Exploratory Data Analysis (EDA) on Methylene Blue

In [ ]:
mb_file = PROCESSED_DIR / "dataset_methylene_blue.csv"
if mb_file.exists():
    df_mb = pd.read_csv(mb_file)
    print(f"Loaded {len(df_mb)} clean rows for Methylene Blue")
    display(df_mb.describe())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(df_mb["removal_pct"], kde=True, ax=axes[0], color="royalblue")
    axes[0].set_title("Distribution of Removal Efficiency (%)")
    
    corr = df_mb[["pH", "dosage_g_L", "conc_mg_L", "time_min", "temp_C", "removal_pct"]].corr()
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=axes[1])
    axes[1].set_title("Feature Correlation Heatmap")
    plt.tight_layout()
    plt.show()

## 3. Predicted vs. Actual Parity Plot

In [ ]:
metrics_file = MODELS_DIR / "methylene_blue" / "metrics.json"
if metrics_file.exists():
    M = json.loads(metrics_file.read_text())
    act = M["test_predictions"]["actual"]
    pred = M["test_predictions"]["predicted"]
    
    plt.figure(figsize=(7, 6))
    plt.scatter(act, pred, color="#10b981", s=60, edgecolors="k", label="Held-out Test Rows")
    plt.plot([0, 100], [0, 100], "r--", label="1:1 Perfect Fit")
    plt.xlabel("Actual Removal (%)")
    plt.ylabel("Predicted Removal (%)")
    plt.title(f"Parity Plot: {M['best']} (Test R² = {M['models'][M['best']]['test_r2']:.3f})")
    plt.legend()
    plt.show()

## 4. Operational Optimization: Dosage Screening Curve

In [ ]:
model_file = MODELS_DIR / "methylene_blue" / "best_model.joblib"
if model_file.exists() and metrics_file.exists():
    model = joblib.load(model_file)
    feats = M["features"]
    lo, hi = M["ranges"]["dosage_g_L"]
    doses = np.linspace(lo, hi, 100)
    
    # Baseline conditions: pH=7.0, conc=50 mg/L, time=120 min, temp=25 C
    sim_data = pd.DataFrame([{
        "pH": 7.0, "dosage_g_L": d, "conc_mg_L": 50.0, "time_min": 120.0, "temp_C": 25.0
    } for d in doses], columns=feats)
    
    preds = np.clip(model.predict(sim_data), 0, 100)
    plt.figure(figsize=(8, 5))
    plt.plot(doses, preds, color="teal", lw=2.5, label="Predicted Removal (%)")
    plt.axhline(85, color="red", ls="--", label="Target: 85% Removal")
    plt.xlabel("PSH Dosage (g/L)")
    plt.ylabel("Predicted Removal Efficiency (%)")
    plt.title("Optimization: Dye Removal vs. PSH Dosage (C0=50 mg/L, pH=7.0)")
    plt.legend()
    plt.show()